# ASL Alphabet — Model Evaluation

In [1]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import torch
import torch.nn as nn
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [2]:
# ── Load data ──────────────────────────────────────────────────────────────
splits = np.load('data/splits.npz')
X_test = splits['X_test']   # shape (9537, 63)
y_test = splits['y_test']   # shape (9537,)  int64  0–27

with open('data/label_to_index.json', 'r') as f:
    label_to_index = json.load(f)

index_to_label = {v: k for k, v in label_to_index.items()}
class_names = [index_to_label[i] for i in range(len(index_to_label))]

# ── Define MLP architecture (must match training) ──────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
    def forward(self, x): return self.net(x)

# ── Load model checkpoint ──────────────────────────────────────────────────
checkpoint = torch.load('models/model.pt', map_location='cpu')
input_dim  = checkpoint['input_dim']    # 63
num_classes = checkpoint['num_classes'] # 28

model = MLP(input_dim, num_classes)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

# ── Run inference ──────────────────────────────────────────────────────────
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    logits = model(X_test_tensor)
    preds = torch.argmax(logits, dim=1).numpy()

test_accuracy = accuracy_score(y_test, preds)
print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Total test samples: {len(y_test)}')
print(f'Number of classes: {num_classes}')
print(f'Class names: {class_names}')

Test accuracy: 0.9845
Total test samples: 9537
Number of classes: 28
Class names: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'space']


In [3]:
# ── Confusion matrix ───────────────────────────────────────────────────────
import os
os.makedirs('docs', exist_ok=True)

cm = confusion_matrix(y_test, preds)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
    linewidths=0.3,
    linecolor='lightgray',
)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix — MLP (test set)', fontsize=15, pad=14)
ax.tick_params(axis='x', labelsize=10, rotation=45)
ax.tick_params(axis='y', labelsize=10, rotation=0)
plt.tight_layout()
plt.savefig('docs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/confusion_matrix.png')

Saved: docs/confusion_matrix.png


C:\Users\Bemnet\AppData\Local\Temp\ipykernel_11816\1333688501.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ── Per-class accuracy ─────────────────────────────────────────────────────
cm_diag = cm.diagonal()
cm_row_sums = cm.sum(axis=1)
per_class_acc = cm_diag / cm_row_sums  # avoid division by zero for classes present in test set

# Build sorted (ascending) lists for plotting
sorted_idx = np.argsort(per_class_acc)
sorted_acc = per_class_acc[sorted_idx]
sorted_labels = [class_names[i] for i in sorted_idx]

overall_acc = test_accuracy
threshold = 0.96
colors = ['tomato' if a < threshold else 'steelblue' for a in sorted_acc]

fig, ax = plt.subplots(figsize=(8, 10))
bars = ax.barh(sorted_labels, sorted_acc, color=colors, edgecolor='white', height=0.7)
ax.axvline(x=overall_acc, color='black', linestyle='--', linewidth=1.4,
           label=f'Overall accuracy: {overall_acc:.4f}')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Per-Class Accuracy — MLP', fontsize=14, pad=12)
ax.legend(fontsize=10)

# Annotate bar values
for bar, acc in zip(bars, sorted_acc):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.3f}', va='center', ha='left', fontsize=8)

# Add legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='tomato', label=f'Accuracy < {threshold}'),
    Patch(facecolor='steelblue', label=f'Accuracy ≥ {threshold}'),
]
ax.legend(handles=legend_elements + [plt.Line2D([0], [0], color='black', linestyle='--',
          label=f'Overall accuracy: {overall_acc:.4f}')],
          fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('docs/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/per_class_accuracy.png')

Saved: docs/per_class_accuracy.png


C:\Users\Bemnet\AppData\Local\Temp\ipykernel_11816\2327102245.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Model comparison table ─────────────────────────────────────────────────
with open('results_mlp.json', 'r') as f:
    results = json.load(f)

mlp_acc   = results['mlp']['test_accuracy']
mlp_time  = results['mlp']['train_time_s']
rf_acc    = results['rf']['test_accuracy']
rf_time   = results['rf']['train_time_s']

webcam_note = 'TBD — capture ~120 frames (5 per letter × 24 static letters)'

header = f"{'Model':<6} | {'Test Accuracy':>14} | {'Train Time (s)':>14} | Webcam Generalization"
sep    = '-' * len(header)
row_mlp = (f"{'MLP':<6} | {mlp_acc:>14.4f} | {mlp_time:>14.1f} | {webcam_note}")
row_rf  = (f"{'RF':<6} | {rf_acc:>14.4f} | {rf_time:>14.1f} | {webcam_note}")

print(sep)
print(header)
print(sep)
print(row_mlp)
print(row_rf)
print(sep)

print()
print(f'Accuracy delta (RF − MLP): {rf_acc - mlp_acc:+.4f}')
print(f'Train-time delta (RF − MLP): {rf_time - mlp_time:+.1f} s')

----------------------------------------------------------------
Model  |  Test Accuracy | Train Time (s) | Webcam Generalization
----------------------------------------------------------------
MLP    |         0.9845 |           45.7 | TBD — capture ~120 frames (5 per letter × 24 static letters)
RF     |         0.9880 |           37.0 | TBD — capture ~120 frames (5 per letter × 24 static letters)
----------------------------------------------------------------

Accuracy delta (RF − MLP): +0.0036
Train-time delta (RF − MLP): -8.7 s


## Error Analysis

Overall the MLP achieves strong performance across the 28-class ASL alphabet, with most letters classified at or above 98% per-class accuracy. The most notable confusion occurs between **M and N**: both signs involve bending the fingers down over the thumb, with M using three fingers and N using two — a subtle distinction that is easily collapsed when landmark coordinates are noisy or the hand is slightly rotated, causing the model to conflate the two. **X** shows comparatively lower accuracy because its defining feature is a hooked or crooked index finger, a shape that can closely resemble a relaxed B or a partially extended D depending on hand orientation and scale; the single-frame landmark snapshot loses the dynamic curling motion that signers use to disambiguate X in practice. **J and Z** are inherently motion-based letters in standard ASL — J traces a hook with the pinky and Z draws a zigzag with the index finger — but in this dataset they are reduced to static pose snapshots (the start or end handshape). As a result, the model learns to recognize those terminal handshapes rather than the motion trajectory, which means webcam performance on real signers who produce the full movement may differ from the held-out test accuracy reported here. For all remaining letters the classifier generalises well, and the confusion matrix shows that most off-diagonal mass is concentrated in a small number of visually similar pairs rather than spread uniformly across the label space.